# 11 — Claim Frequency Tuning

## Objectif

L'étape précédente a montré que XGBoost Poisson améliore nettement
la baseline constante et PoissonRegressor.

Résultats initiaux :

- Constant Frequency : Poisson Deviance = 0.330945
- PoissonRegressor : Poisson Deviance = 0.329097
- XGBoost Poisson : Poisson Deviance = 0.303724

Cette étape vise à :

1. ajuster la régularisation de PoissonRegressor ;
2. ajuster les hyperparamètres de XGBoost Poisson ;
3. comparer les modèles avec le même protocole ;
4. analyser la calibration de fréquence ;
5. verrouiller un candidat avant le test final.

Le jeu de test n'est pas utilisé pendant cette étape.

In [1]:
import pandas as pd
import numpy as np

import json
import time

from pathlib import Path

from sklearn.base import clone
from sklearn.model_selection import (
    StratifiedKFold,
    ParameterSampler
)

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

from sklearn.impute import SimpleImputer
from sklearn.linear_model import PoissonRegressor

from sklearn.metrics import (
    mean_poisson_deviance,
    mean_absolute_error
)

from xgboost import XGBRegressor


RANDOM_STATE = 42

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [2]:
cwd = Path.cwd()

possible_processed_paths = [
    cwd / "data" / "processed",
    cwd / "data-science" / "data" / "processed",
    cwd.parent / "data" / "processed"
]

PROCESSED_DIR = None

for path in possible_processed_paths:
    if path.exists():
        PROCESSED_DIR = path
        break

if PROCESSED_DIR is None:
    raise FileNotFoundError(
        "Impossible de localiser data/processed"
    )


DATA_DIR = PROCESSED_DIR.parent
METADATA_DIR = DATA_DIR / "metadata"

REPORT_DIR = (
    DATA_DIR.parent
    / "reports"
    / "models"
    / "claim_frequency"
    / "tuning"
)

REPORT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


frequency = pd.read_csv(
    PROCESSED_DIR
    / "claim_frequency_modeling_base.csv"
)

split_df = pd.read_csv(
    METADATA_DIR
    / "freMTPL2_split.csv"
)


In [3]:
frequency = frequency.merge(
    split_df,
    on="IDpol",
    how="left",
    validate="one_to_one"
)

frequency_dev = (
    frequency[
        frequency["split"]
        == "development"
    ]
    .copy()
)

frequency_test = (
    frequency[
        frequency["split"]
        == "test"
    ]
    .copy()
)

print(
    "DEV :",
    frequency_dev.shape
)

print(
    "TEST :",
    frequency_test.shape
)

DEV : (541431, 13)
TEST : (135358, 13)


In [4]:
with open(
    METADATA_DIR
    / "feature_manifest.json",
    "r",
    encoding="utf-8"
) as f:

    feature_manifest = json.load(f)


config = feature_manifest[
    "claim_frequency"
]

numeric_features = config[
    "numeric_features"
]

categorical_features = config[
    "categorical_features"
]

features = (
    numeric_features
    +
    categorical_features
)


X_dev = (
    frequency_dev[
        features
    ]
    .copy()
)

y_count_dev = (
    frequency_dev[
        "ClaimNb"
    ]
    .copy()
)

exposure_dev = (
    frequency_dev[
        "Exposure"
    ]
    .copy()
)

y_rate_dev = (
    y_count_dev
    /
    exposure_dev
)

strata_dev = (
    y_count_dev > 0
).astype(int)

In [5]:
linear_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        ),
        (
            "scaler",
            StandardScaler()
        )
    ]
)


tree_numeric_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="median"
            )
        )
    ]
)


categorical_pipeline = Pipeline(
    steps=[
        (
            "imputer",
            SimpleImputer(
                strategy="most_frequent"
            )
        ),
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)


linear_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            linear_numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)


tree_preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            tree_numeric_pipeline,
            numeric_features
        ),
        (
            "categorical",
            categorical_pipeline,
            categorical_features
        )
    ]
)

In [6]:
tuning_cv = StratifiedKFold(
    n_splits=3,
    shuffle=True,
    random_state=RANDOM_STATE
)

In [7]:
def evaluate_frequency_pipeline(
    pipeline,
    X,
    y_count,
    exposure,
    strata,
    cv
):

    fold_deviances = []
    fold_mae = []
    observed_frequencies = []
    predicted_frequencies = []

    for train_idx, val_idx in cv.split(
        X,
        strata
    ):

        X_train = X.iloc[
            train_idx
        ]

        X_val = X.iloc[
            val_idx
        ]

        y_train_count = y_count.iloc[
            train_idx
        ]

        y_val_count = y_count.iloc[
            val_idx
        ]

        exposure_train = exposure.iloc[
            train_idx
        ]

        exposure_val = exposure.iloc[
            val_idx
        ]

        y_train_rate = (
            y_train_count
            /
            exposure_train
        )

        model = clone(
            pipeline
        )

        model.fit(
            X_train,
            y_train_rate,
            model__sample_weight=(
                exposure_train
            )
        )

        predicted_rate = model.predict(
            X_val
        )

        predicted_rate = np.clip(
            predicted_rate,
            1e-9,
            None
        )

        predicted_count = (
            predicted_rate
            *
            exposure_val
        )

        deviance = mean_poisson_deviance(
            y_val_count,
            predicted_count
        )

        mae = mean_absolute_error(
            y_val_count,
            predicted_count
        )

        observed_frequency = (
            y_val_count.sum()
            /
            exposure_val.sum()
        )

        predicted_frequency = (
            predicted_count.sum()
            /
            exposure_val.sum()
        )

        fold_deviances.append(
            deviance
        )

        fold_mae.append(
            mae
        )

        observed_frequencies.append(
            observed_frequency
        )

        predicted_frequencies.append(
            predicted_frequency
        )


    return {
        "poisson_deviance_mean":
            np.mean(
                fold_deviances
            ),

        "poisson_deviance_std":
            np.std(
                fold_deviances,
                ddof=1
            ),

        "count_mae_mean":
            np.mean(
                fold_mae
            ),

        "observed_frequency_mean":
            np.mean(
                observed_frequencies
            ),

        "predicted_frequency_mean":
            np.mean(
                predicted_frequencies
            )
    }

In [8]:
poisson_alpha_values = [
    0.0001,
    0.001,
    0.01,
    0.1,
    0.5,
    1.0,
    2.0,
    5.0,
    10.0
]

In [9]:
poisson_tuning_results = []

for alpha in poisson_alpha_values:

    print(
        "Testing alpha =",
        alpha
    )

    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                linear_preprocessor
            ),
            (
                "model",
                PoissonRegressor(
                    alpha=alpha,
                    max_iter=1000
                )
            )
        ]
    )

    result = evaluate_frequency_pipeline(
        pipeline=pipeline,
        X=X_dev,
        y_count=y_count_dev,
        exposure=exposure_dev,
        strata=strata_dev,
        cv=tuning_cv
    )

    result[
        "alpha"
    ] = alpha

    poisson_tuning_results.append(
        result
    )

Testing alpha = 0.0001
Testing alpha = 0.001
Testing alpha = 0.01
Testing alpha = 0.1
Testing alpha = 0.5
Testing alpha = 1.0
Testing alpha = 2.0
Testing alpha = 5.0
Testing alpha = 10.0


In [10]:
poisson_tuning_df = (
    pd.DataFrame(
        poisson_tuning_results
    )
    .sort_values(
        "poisson_deviance_mean"
    )
    .reset_index(
        drop=True
    )
)

poisson_tuning_df

,poisson_deviance_mean,poisson_deviance_std,count_mae_mean,observed_frequency_mean,predicted_frequency_mean,alpha
0,0.320641,0.000597,0.098825,0.100923,0.100876,0.0010
1,0.320648,0.000575,0.098834,0.100923,0.100927,0.0001
2,0.320791,0.000651,0.098994,0.100923,0.100983,0.0100
3,0.323300,0.000897,0.099462,0.100923,0.100909,0.1000
4,0.327731,0.001125,0.099804,0.100923,0.100928,0.5000
5,0.329099,0.001171,0.099871,0.100923,0.100926,1.0000
6,0.329950,0.001197,0.099908,0.100923,0.100927,2.0000
7,0.330527,0.001212,0.099931,0.100923,0.100928,5.0000
8,0.330732,0.001217,0.099894,0.100923,0.100831,10.0000


In [11]:
best_poisson_row = (
    poisson_tuning_df
    .iloc[0]
)

BEST_POISSON_ALPHA = float(
    best_poisson_row[
        "alpha"
    ]
)

print(
    "Best Poisson alpha :",
    BEST_POISSON_ALPHA
)

print(
    "Best Poisson deviance :",
    best_poisson_row[
        "poisson_deviance_mean"
    ]
)

Best Poisson alpha : 0.001
Best Poisson deviance : 0.32064069278385143


In [12]:
poisson_tuning_df.to_csv(
    REPORT_DIR
    / "poisson_tuning_results.csv",
    index=False
)

In [13]:
xgb_param_space = {

    "n_estimators": [
        150,
        200,
        300,
        500,
        700
    ],

    "learning_rate": [
        0.02,
        0.03,
        0.05,
        0.08,
        0.10
    ],

    "max_depth": [
        3,
        4,
        5,
        6,
        7
    ],

    "min_child_weight": [
        1,
        3,
        5,
        10
    ],

    "subsample": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "colsample_bytree": [
        0.7,
        0.8,
        0.9,
        1.0
    ],

    "reg_alpha": [
        0.0,
        0.01,
        0.1,
        0.5,
        1.0
    ],

    "reg_lambda": [
        1.0,
        2.0,
        5.0,
        10.0
    ]
}

In [14]:
N_ITER_XGB = 20

sampled_parameters = list(
    ParameterSampler(
        xgb_param_space,
        n_iter=N_ITER_XGB,
        random_state=RANDOM_STATE
    )
)

len(
    sampled_parameters
)

20

In [15]:
xgb_tuning_results = []

start_total = time.perf_counter()


for i, params in enumerate(
    sampled_parameters,
    start=1
):

    print(
        f"\nConfiguration {i}/{N_ITER_XGB}"
    )

    print(
        params
    )


    xgb_model = XGBRegressor(
        objective="count:poisson",
        eval_metric="poisson-nloglik",
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
        **params
    )


    pipeline = Pipeline(
        steps=[
            (
                "preprocessor",
                tree_preprocessor
            ),
            (
                "model",
                xgb_model
            )
        ]
    )


    start_config = (
        time.perf_counter()
    )


    result = evaluate_frequency_pipeline(
        pipeline=pipeline,
        X=X_dev,
        y_count=y_count_dev,
        exposure=exposure_dev,
        strata=strata_dev,
        cv=tuning_cv
    )


    elapsed = (
        time.perf_counter()
        -
        start_config
    )


    result.update(
        params
    )

    result[
        "elapsed_seconds"
    ] = elapsed


    xgb_tuning_results.append(
        result
    )


xgb_tuning_total_time = (
    time.perf_counter()
    -
    start_total
)


Configuration 1/20
{'subsample': 0.9, 'reg_lambda': 2.0, 'reg_alpha': 0.1, 'n_estimators': 700, 'min_child_weight': 1, 'max_depth': 4, 'learning_rate': 0.02, 'colsample_bytree': 1.0}

Configuration 2/20
{'subsample': 1.0, 'reg_lambda': 1.0, 'reg_alpha': 1.0, 'n_estimators': 150, 'min_child_weight': 10, 'max_depth': 4, 'learning_rate': 0.08, 'colsample_bytree': 1.0}

Configuration 3/20
{'subsample': 0.7, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 700, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.03, 'colsample_bytree': 1.0}

Configuration 4/20
{'subsample': 0.9, 'reg_lambda': 10.0, 'reg_alpha': 0.0, 'n_estimators': 200, 'min_child_weight': 10, 'max_depth': 7, 'learning_rate': 0.05, 'colsample_bytree': 0.9}

Configuration 5/20
{'subsample': 1.0, 'reg_lambda': 2.0, 'reg_alpha': 0.1, 'n_estimators': 500, 'min_child_weight': 10, 'max_depth': 7, 'learning_rate': 0.1, 'colsample_bytree': 0.9}

Configuration 6/20
{'subsample': 0.7, 'reg_lambda': 10.0, 'reg_alpha': 0.01

In [16]:
xgb_tuning_df = (
    pd.DataFrame(
        xgb_tuning_results
    )
    .sort_values(
        "poisson_deviance_mean"
    )
    .reset_index(
        drop=True
    )
)

In [17]:
xgb_tuning_df[
    [
        "poisson_deviance_mean",
        "poisson_deviance_std",
        "count_mae_mean",
        "predicted_frequency_mean",
        "n_estimators",
        "learning_rate",
        "max_depth",
        "min_child_weight",
        "subsample",
        "colsample_bytree",
        "reg_alpha",
        "reg_lambda",
        "elapsed_seconds"
    ]
].head(
    10
)

,poisson_deviance_mean,poisson_deviance_std,count_mae_mean,predicted_frequency_mean,n_estimators,learning_rate,max_depth,min_child_weight,subsample,colsample_bytree,reg_alpha,reg_lambda,elapsed_seconds
0,0.302858,0.001448,0.096058,0.100953,500,0.08,6,10,0.7,0.9,0.01,10.0,59.904027
1,0.302973,0.001439,0.096306,0.100791,500,0.02,7,5,0.9,0.9,0.50,5.0,60.284617
2,0.303114,0.001427,0.096358,0.100863,200,0.05,7,10,0.9,0.9,0.00,10.0,27.832591
3,0.303141,0.001369,0.096350,0.100863,200,0.05,7,5,1.0,0.9,1.00,5.0,25.900862
4,0.303235,0.001673,0.095871,0.100978,500,0.10,7,10,1.0,0.9,0.10,2.0,46.424092
5,0.303359,0.001468,0.096420,0.100913,700,0.02,6,10,0.7,1.0,0.00,2.0,79.920288
6,0.303573,0.001474,0.096434,0.100899,700,0.03,5,3,0.7,1.0,0.00,10.0,78.117101
7,0.303706,0.001415,0.096603,0.100551,200,0.03,7,3,0.9,0.8,0.00,2.0,29.008890
8,0.304207,0.001502,0.096704,0.100896,150,0.08,5,1,0.8,0.8,0.01,2.0,19.480266
9,0.304493,0.001479,0.096806,0.100859,200,0.05,5,3,1.0,0.8,0.50,2.0,20.099080


In [18]:
best_xgb_row = (
    xgb_tuning_df
    .iloc[0]
)

BEST_XGB_PARAMS = {
    "n_estimators":
        int(
            best_xgb_row[
                "n_estimators"
            ]
        ),

    "learning_rate":
        float(
            best_xgb_row[
                "learning_rate"
            ]
        ),

    "max_depth":
        int(
            best_xgb_row[
                "max_depth"
            ]
        ),

    "min_child_weight":
        float(
            best_xgb_row[
                "min_child_weight"
            ]
        ),

    "subsample":
        float(
            best_xgb_row[
                "subsample"
            ]
        ),

    "colsample_bytree":
        float(
            best_xgb_row[
                "colsample_bytree"
            ]
        ),

    "reg_alpha":
        float(
            best_xgb_row[
                "reg_alpha"
            ]
        ),

    "reg_lambda":
        float(
            best_xgb_row[
                "reg_lambda"
            ]
        )
}

In [19]:
print(
    "BEST XGBOOST PARAMETERS"
)

for key, value in (
    BEST_XGB_PARAMS.items()
):

    print(
        key,
        "=",
        value
    )


print(
    "\nBest CV Poisson Deviance :",
    best_xgb_row[
        "poisson_deviance_mean"
    ]
)

BEST XGBOOST PARAMETERS
n_estimators = 500
learning_rate = 0.08
max_depth = 6
min_child_weight = 10.0
subsample = 0.7
colsample_bytree = 0.9
reg_alpha = 0.01
reg_lambda = 10.0

Best CV Poisson Deviance : 0.3028578547775393


In [20]:
xgb_tuning_df.to_csv(
    REPORT_DIR
    / "xgboost_frequency_tuning_results.csv",
    index=False
)

In [21]:
tuning_comparison = pd.DataFrame([
    {
        "model":
            "PoissonRegressor Tuned",

        "poisson_deviance":
            best_poisson_row[
                "poisson_deviance_mean"
            ],

        "poisson_deviance_std":
            best_poisson_row[
                "poisson_deviance_std"
            ],

        "count_mae":
            best_poisson_row[
                "count_mae_mean"
            ],

        "predicted_frequency":
            best_poisson_row[
                "predicted_frequency_mean"
            ]
    },

    {
        "model":
            "XGBoost Poisson Tuned",

        "poisson_deviance":
            best_xgb_row[
                "poisson_deviance_mean"
            ],

        "poisson_deviance_std":
            best_xgb_row[
                "poisson_deviance_std"
            ],

        "count_mae":
            best_xgb_row[
                "count_mae_mean"
            ],

        "predicted_frequency":
            best_xgb_row[
                "predicted_frequency_mean"
            ]
    }
])

tuning_comparison

,model,poisson_deviance,poisson_deviance_std,count_mae,predicted_frequency
0,PoissonRegressor Tuned,0.320641,0.000597,0.098825,0.100876
1,XGBoost Poisson Tuned,0.302858,0.001448,0.096058,0.100953


In [22]:
frequency_candidate = {

    "task":
        "claim_frequency",

    "model":
        "xgboost_poisson",

    "best_parameters":
        BEST_XGB_PARAMS,

    "best_cv_poisson_deviance":
        float(
            best_xgb_row[
                "poisson_deviance_mean"
            ]
        ),

    "best_cv_poisson_deviance_std":
        float(
            best_xgb_row[
                "poisson_deviance_std"
            ]
        ),

    "predicted_frequency_mean":
        float(
            best_xgb_row[
                "predicted_frequency_mean"
            ]
        ),

    "exposure_strategy":
        "target_rate_with_exposure_sample_weight",

    "test_used":
        False
}

In [23]:
with open(
    METADATA_DIR
    / "claim_frequency_candidate.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        frequency_candidate,
        f,
        indent=4,
        ensure_ascii=False
    )

In [24]:
print(
    "=" * 80
)

print(
    "CLAIM FREQUENCY — TUNING RESULTS"
)

print(
    "=" * 80
)


print(
    "\n=== POISSON REGRESSOR ==="
)

print(
    "Best alpha :",
    BEST_POISSON_ALPHA
)

print(
    "Best Poisson Deviance :",
    best_poisson_row[
        "poisson_deviance_mean"
    ]
)

print(
    "Std :",
    best_poisson_row[
        "poisson_deviance_std"
    ]
)

print(
    "Count MAE :",
    best_poisson_row[
        "count_mae_mean"
    ]
)

print(
    "Predicted frequency :",
    best_poisson_row[
        "predicted_frequency_mean"
    ]
)


print(
    "\n=== XGBOOST POISSON ==="
)

for key, value in (
    BEST_XGB_PARAMS.items()
):

    print(
        key,
        "=",
        value
    )

print(
    "Best Poisson Deviance :",
    best_xgb_row[
        "poisson_deviance_mean"
    ]
)

print(
    "Std :",
    best_xgb_row[
        "poisson_deviance_std"
    ]
)

print(
    "Count MAE :",
    best_xgb_row[
        "count_mae_mean"
    ]
)

print(
    "Observed frequency :",
    best_xgb_row[
        "observed_frequency_mean"
    ]
)

print(
    "Predicted frequency :",
    best_xgb_row[
        "predicted_frequency_mean"
    ]
)


print(
    "\n=== TOP 10 XGBOOST CONFIGURATIONS ==="
)

print(
    xgb_tuning_df
    .head(10)
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\n=== FINAL COMPARISON ==="
)

print(
    tuning_comparison
    .round(6)
    .to_string(
        index=False
    )
)


print(
    "\nXGBoost tuning total time :",
    xgb_tuning_total_time
)


print(
    "\nTEST FREQUENCY UTILISÉ ? NON"
)

print(
    "=" * 80
)

CLAIM FREQUENCY — TUNING RESULTS

=== POISSON REGRESSOR ===
Best alpha : 0.001
Best Poisson Deviance : 0.32064069278385143
Std : 0.0005966894403366636
Count MAE : 0.09882544757624168
Predicted frequency : 0.10087581614220538

=== XGBOOST POISSON ===
n_estimators = 500
learning_rate = 0.08
max_depth = 6
min_child_weight = 10.0
subsample = 0.7
colsample_bytree = 0.9
reg_alpha = 0.01
reg_lambda = 10.0
Best Poisson Deviance : 0.3028578547775393
Std : 0.001447858816514848
Count MAE : 0.09605792484108787
Observed frequency : 0.10092290997013376
Predicted frequency : 0.10095271249770155

=== TOP 10 XGBOOST CONFIGURATIONS ===
 poisson_deviance_mean  poisson_deviance_std  count_mae_mean  observed_frequency_mean  predicted_frequency_mean  subsample  reg_lambda  reg_alpha  n_estimators  min_child_weight  max_depth  learning_rate  colsample_bytree  elapsed_seconds
              0.302858              0.001448        0.096058                 0.100923                  0.100953        0.7        10.0 

# Analyse du tuning — Claim Frequency

## PoissonRegressor

La recherche sur le paramètre de régularisation `alpha` sélectionne :

**alpha = 0.001**

Les performances obtenues en validation croisée sont :

- Mean Poisson Deviance : **0.320641**
- Écart-type : **0.000597**
- Count MAE : **0.098825**
- Fréquence moyenne prédite : **0.100876**

Le faible écart-type indique une bonne stabilité des résultats
entre les folds.

PoissonRegressor constitue une baseline actuarielle utile et
interprétable, mais ses performances restent inférieures à celles
obtenues avec XGBoost Poisson.

---

## XGBoost Poisson

La meilleure configuration identifiée est :

- `n_estimators = 500`
- `learning_rate = 0.08`
- `max_depth = 6`
- `min_child_weight = 10`
- `subsample = 0.7`
- `colsample_bytree = 0.9`
- `reg_alpha = 0.01`
- `reg_lambda = 10`

Elle obtient :

- Mean Poisson Deviance : **0.302858**
- Écart-type : **0.001448**
- Count MAE : **0.096058**
- Fréquence observée moyenne : **0.100923**
- Fréquence prédite moyenne : **0.100953**

La fréquence globale prédite est extrêmement proche de la fréquence
réellement observée.

Le biais relatif de fréquence est d'environ :

**+0.03 %**

Le modèle reste donc correctement calibré au niveau global du portefeuille.

---

## Comparaison avec PoissonRegressor

Sur le même protocole de tuning à trois folds :

PoissonRegressor :

**0.320641**

XGBoost Poisson :

**0.302858**

XGBoost réduit ainsi la Mean Poisson Deviance d'environ :

**5.55 %**

par rapport au meilleur PoissonRegressor testé.

Cela confirme l'intérêt d'un modèle non linéaire capable de représenter
des interactions et relations plus complexes entre les caractéristiques
du contrat et la fréquence de sinistre.

---

## Stabilité et sélection

L'écart-type de la Poisson deviance de XGBoost est :

**0.001448**

ce qui reste faible relativement à la performance moyenne.

Plusieurs configurations XGBoost obtiennent néanmoins des performances
très proches.

Par exemple, les trois meilleures Poisson Deviances sont environ :

- 0.302858
- 0.302973
- 0.303114

La configuration retenue ne doit donc pas être considérée comme
universellement optimale.

Elle correspond simplement à la meilleure configuration parmi celles
évaluées selon le protocole expérimental défini.

---

## Décision

XGBoost Poisson est retenu comme candidat final pour le modèle
de fréquence de sinistre.

PoissonRegressor avec `alpha = 0.001` est conservé comme baseline
actuarielle de comparaison.

La configuration XGBoost est maintenant verrouillée avant l'évaluation
du holdout Frequency.

Aucune modification des hyperparamètres ne sera effectuée sur la base
des résultats du holdout final.